In [ ]:
!pip install panel bokeh jupyter_bokeh openpyxl scipy numpy pandas

In [16]:
import numpy as np
import pandas as pd
import panel as pn
from scipy.special import erfcinv
from bokeh.plotting import figure
from bokeh.models import LogColorMapper, ColorBar, HoverTool
from bokeh.transform import transform

# Ensure Jupyter Bokeh is registered
pn.extension('bokeh', sizing_mode="stretch_width")

# Physics & Constants
R_GAS = 8.3144626e-3    # kJ mol^-1 K^-1
C_TARGET = 5.0e14       # Target impurity concentration (cm^-3)
N_AVO = 6.02214076e23
DAY = 86400.0

# Default Alloys and Atomic Masses
ALLOYS = {
    "Rene41": dict(
        density=8.249,
        comp={"Ni": 49.2585, "Fe": 5.0, "Cr": 19.0, "Mo": 9.75, "Co": 11.0,
              "Ti": 3.15, "Al": 1.6, "Mn": 0.1, "Si": 0.5, "Cu": 0.5,
              "C": 0.12, "S": 0.015, "B": 0.0065},
    ),
    "Inconel718": dict(
        density=8.26,
        comp={"Ni": 52.5, "Fe": 19.0, "Cr": 19.0, "Nb": 5.125, "Mo": 3.05,
              "Co": 1.0, "Ti": 0.9, "Al": 0.5, "Mn": 0.35, "Si": 0.35,
              "Cu": 0.3, "C": 0.08, "Ta": 0.05, "P": 0.015, "S": 0.015,
              "B": 0.006},
    ),
    "Inconel625": dict(
        density=8.44,
        comp={"Ni": 58.6, "Fe": 5.0, "Cr": 22.0, "Nb": 1.75, "Mo": 9.0,
              "Co": 1.0, "Al": 0.4, "Mn": 0.5, "Ta": 1.75},
    ),
    "Haynes282": dict(
        density=8.28,
        comp={"Ni": 57.0, "Fe": 1.5, "Cr": 20.0, "Mo": 8.5, "Co": 10.0,
              "Ti": 2.1, "Al": 1.5, "Mn": 0.3, "Si": 0.15, "C": 0.06, "B": 0.005},
    ),
    "1.4571": dict(
        density=8.0,
        comp={"Fe": 0.67, "Cr": 0.17, "Ni": 0.12, "Mo": 0.022,
              "Mn": 0.015, "Ti": 0.005, "Si": 0.005, "C": 0.0005},
    ),
    "1.4435": dict(
        density=8.0,
        comp={"Fe": 0.65, "Cr": 0.175, "Ni": 0.13, "Mo": 0.027,
              "Mn": 0.015, "Si": 0.005, "C": 0.0003},
    )
}

ATOMIC_MASS = {
    "Ni": 58.69, "Fe": 55.85, "Cr": 52.00, "Nb": 92.91, "Mo": 95.95,
    "Co": 58.93, "Ti": 47.87, "Al": 26.98, "Mn": 54.94, "Si": 28.09,
    "Cu": 63.55, "C":  12.01, "Ta": 180.95, "P": 30.97, "S": 32.07,
    "B":  10.81, "W": 183.84, "Ag": 107.87
}

# Hardcoded diffusion parameters (D0 in m2/s, Q in kJ/mol)
DIFF_PARAMS = {
    'B': (2.11e-05, 163.38),
    'C': (3.54e-05, 173.73),
    'Al': (2.43e-05, 166.2),
    'Si': (5.4e-05, 182.15),
    'P': (2e-05, 162.32),
    'S': (2.29e-05, 165.06),
    'Sc': (0.000139, 201.02),
    'Ti': (0.000175, 205.54),
    'V': (0.000275, 214.56),
    'Cr': (0.000282, 215.09),
    'Mn': (0.000105, 195.33),
    'Fe': (0.000211, 209.33),
    'Co': (0.000191, 207.34),
    'Ni': (0.000195, 207.77),
    'Cu': (7.69e-05, 189.19),
    'Zn': (2.81e-05, 169.14),
    'Ga': (2.67e-05, 168.1),
    'Y': (0.000145, 201.76),
    'Zr': (0.000407, 221.46),
    'Nb': (0.000438, 222.84),
    'Mo': (0.000572, 227.42),
    'La': (8.4e-05, 191.06),
    'Hf': (0.000274, 214.46),
    'W': (0.00197, 253.74)
}

print(f"Successfully loaded self-contained diffusion parameters for elements: {list(DIFF_PARAMS.keys())}")

# Widgets
oy_options = list(ALLOYS.keys())
alloy_sel = pn.widgets.Select(name="Alloy", options=oy_options, value="Rene41")
temp_max_w = pn.widgets.FloatSlider(name="Max Temperature (°C)", start=100, end=1200, step=50, value=800)
thick_max_w = pn.widgets.FloatSlider(name="Max Thickness (mm)", start=0.1, end=2.0, step=0.1, value=1.5)
el_sel = pn.widgets.Select(name="Element", options=list(DIFF_PARAMS.keys()), value=list(DIFF_PARAMS.keys())[0])

@pn.depends(alloy_sel, temp_max_w, thick_max_w, el_sel)
def build_heatmap_view(alloy, temp_max, thick_max, el):
    if not el or el not in DIFF_PARAMS:
        return pn.pane.Markdown("### Selected element data is unavailable.")

    D0, Q = DIFF_PARAMS[el]
    num_pts = 60
    t_c_range = np.linspace(1, temp_max, num_pts)
    thick_mm_range = np.linspace(0.01, thick_max, num_pts)

    T_K, L_mm = np.meshgrid(t_c_range + 273.15, thick_mm_range)
    L_m = L_mm * 1e-3

    alloy_data = ALLOYS[alloy]
    w = alloy_data["comp"].get(el, 0.0)
    if w <= 0 or el not in ATOMIC_MASS:
        return pn.pane.Markdown(f"### Element {el} is not present in the {alloy} composition!")

    C0 = (w / 100.0) * alloy_data["density"] * N_AVO / ATOMIC_MASS[el]
    if C0 <= C_TARGET:
        return pn.pane.Markdown(f"### Source concentration ({C0:.2e} cm⁻³) is below target ({C_TARGET:.2e} cm⁻³).")

    D = D0 * np.exp(-Q / (R_GAS * T_K))
    y_val = erfcinv(C_TARGET / C0)
    t_seconds = (L_m ** 2) / (4.0 * D * y_val ** 2)
    t_days = t_seconds / DAY

    # Prevent log(0) or negative values in LogColorMapper by setting a positive floor
    t_days = np.clip(t_days, 1e-5, None)

    x_coords = []
    y_coords = []
    values = []
    for i in range(num_pts):
        for j in range(num_pts):
            x_coords.append(t_c_range[j])
            y_coords.append(thick_mm_range[i])
            values.append(t_days[i, j])

    source_data = pd.DataFrame({
        'temperature': x_coords,
        'thickness': y_coords,
        'days': values
    })

    colors = ["#313695", "#4575b4", "#74add1", "#abd9e9", "#e0f3f8", "#fee090", "#fdae61", "#f46d43", "#d73027", "#a50026"]
    mapper = LogColorMapper(palette=colors, low=float(np.nanmin(values)), high=float(np.nanmax(values)))

    p = figure(
        title=f"Time to reach {C_TARGET:.1e} cm⁻³ (days) for {el} diffusion",
        x_axis_label="Temperature (°C)",
        y_axis_label="Thickness (mm)",
        width=600,
        height=450,
        tools="pan,box_zoom,reset,save"
    )

    p.rect(
        x='temperature',
        y='thickness',
        width=temp_max/num_pts,
        height=thick_max/num_pts,
        source=source_data,
        fill_color=transform('days', mapper),
        line_color=None
    )

    color_bar = ColorBar(color_mapper=mapper, label_standoff=12, border_line_color=None, location=(0,0))
    p.add_layout(color_bar, 'right')

    hover = HoverTool(tooltips=[
        ("Temperature", "@temperature{0.0} °C"),
        ("Thickness", "@thickness{0.00} mm"),
        ("Time", "@days{0.00e} days")
    ])
    p.add_tools(hover)
    return p

heatmap_app = pn.Column(
    pn.pane.Markdown("# Ag Barrier Diffusion - Interactive 2D Heatmap"),
    pn.Row(alloy_sel, el_sel),
    pn.Row(temp_max_w, thick_max_w),
    pn.layout.Divider(),
    build_heatmap_view
)

heatmap_app

/tmp/ipykernel_622/4154129699.py:10: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension('bokeh', sizing_mode="stretch_width")


Successfully loaded self-contained diffusion parameters for elements: ['B', 'C', 'Al', 'Si', 'P', 'S', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Y', 'Zr', 'Nb', 'Mo', 'La', 'Hf', 'W']


Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Row(sizing_mode='stretch_width')
        [0] Select(label='Alloy', name='Alloy', options=['Rene41', 'Inconel718', ...], sizing_mode='stretch_width', value='Rene41')
        [1] Select(label='Element', name='Element', options=['B', 'C', 'Al', ...], sizing_mode='stretch_width', value='B')
    [2] Row(sizing_mode='stretch_width')
        [0] FloatSlider(end=1200, label='Max Temperature (°C)', name='Max Temperature (°C)', sizing_mode='stretch_width', start=100, step=50, value=800)
        [1] FloatSlider(end=2.0, label='Max Thickness (mm)', name='Max Thickness (mm)', sizing_mode='stretch_width', start=0.1, value=1.5)
    [3] Divider(sizing_mode='stretch_width')
    [4] ParamFunction(function, _pane=Bokeh, defer_load=False, sizing_mode='stretch_width')